# 记忆：虚拟上下文和MemGPT

上下文窗口是有限的，但是会话、文档以及工具轨迹不是。MemGPT将其类比到OS 的虚拟内存———— 主上下文是RAM，外部存储是磁盘，agent在两者之间换页。

## 问题描述

上下文窗口看起来是能够解决记忆问题，但是并没有，生产中涌现出三种失败模式：
- 溢出。 多轮会话、长文档、或者工具调用很重的轨迹，被窗口截断的内容就消失了。
- 稀释。 即使是在窗口内，填充不相关的内容会稀释掉真正重要的东西，即使是前沿的模型也在长输入下退化。
- 持久化。  新的会话开启一个空窗口。没有外部记忆的agent不能够跨会话回答“你还记得上次....”。

## 基本概念

### MemGPT： 类比OS

将上下文管理类比到操作系统的虚拟内存：
|OS概念|MemGPT概念|2026生成分类|
|---|---|---|
|内存|主上下文（提示词）|上下文窗口|
|磁盘|外部上下文|向量数据库、KV、图存储|
|缺页|调用记忆工具|`memory.search/read/write`|
|OS内核|agent 控制流|带记忆工具的ReAct循环|

一个跑着普通ReAct循环的agent，外带一系列工具让页数据从主窗口流入和流出。

### 两级

- 主上下文。 固定大小，对模型始终可见。
- 外部上下文。 不设限、可以被检索。在相关时读取、事实涌现时写入。

### 中断模式

MemGPT引入了“记忆即中断”————会话中agent可以调用一个记忆工具，运行时执行它，结果作为一项新的观察拼进下一个助手轮。概念上等同于一次Unit`read()`调用，阻塞进程、返回字节、然后进程继续。

典型的记忆工具接口
```
core_memory_append(section, text) 写入prompt的某个持久section
core_memory_replace(section, old, new) 编辑一个持久节
archival_memory_insert(text) 写入可搜索的外部存储
archival_memory_search(query, top_k) 从外部存储检索
conversation_search(query) 扫描过往轮次
```

### 这个模式在哪里会是出错

- 记忆腐烂。  写积累的速度大于读，检索淹没在过时事实里。修法：周期性整合、显式失效。

- 记忆投毒。  外部记忆是检索回来的文本。

- 引用丢失。  agent回忆起某项事实，但说不出是哪一轮。修法：写入的时候带上会话和轮次ID。

# 开始编码

对应本章核心：**两级上下文（主 / 外部）**、**记忆即中断（缺页）**、**五件记忆工具**。  
先用脚本化玩具跑通换页；再用 **PyTorch 小检索器** 示意 archival 向量外存；最后用 **LangChain + DeepSeek** 挂上真实记忆工具循环。


## 1. 教学玩具：MemGPT 换页骨架

- **主上下文**：固定容量的 `persona` / `human` + 有限对话窗口（RAM）。
- **外部上下文**：`ArchivalStore` + 完整 `ConversationLog`（磁盘）。
- **缺页**：主窗口里没有事实时，调用 `archival_memory_search` / `conversation_search`，观察拼回下一轮。


In [ ]:
from __future__ import annotations

import re
from dataclasses import dataclass, field
from typing import Any, Callable

from typing_extensions import TypedDict


@dataclass
class MemoryEntry:
    """外部档案中的一条记忆（带会话/轮次引用，防引用丢失）。"""

    text: str
    session_id: str
    turn_id: int
    entry_id: int


@dataclass
class CoreMemory:
    """主上下文中的持久节（始终可见）。"""

    sections: dict[str, str] = field(
        default_factory=lambda: {"persona": "You are a helpful assistant.", "human": ""}
    )

    def append(self, section: str, text: str) -> str:
        """
        Args:
            section: 节名（如 ``persona`` / ``human``）。
            text: 追加文本。

        Returns:
            observation: 执行结果。
        """
        if section not in self.sections:
            self.sections[section] = ""
        sep = "" if not self.sections[section] else " "
        self.sections[section] = (self.sections[section] + sep + text).strip()
        return f"OK: appended to core[{section}] (len={len(self.sections[section])})"

    def replace(self, section: str, old: str, new: str) -> str:
        """
        Args:
            section: 节名。
            old: 旧子串。
            new: 新子串。

        Returns:
            observation: 成功或未找到。
        """
        if section not in self.sections:
            return f"Error: unknown section {section}"
        if old not in self.sections[section]:
            return f"Error: old text not found in core[{section}]"
        self.sections[section] = self.sections[section].replace(old, new, 1)
        return f"OK: replaced in core[{section}]"

    def render(self) -> str:
        """
        Returns:
            block: 可注入主提示的核心记忆块。
        """
        lines = ["# Core memory (always in RAM)"]
        for k, v in self.sections.items():
            lines.append(f"## {k}\n{v or '<empty>'}")
        return "\n".join(lines)


@dataclass
class ArchivalStore:
    """外部可搜索档案（磁盘）；玩具版用词重叠打分。"""

    entries: list[MemoryEntry] = field(default_factory=list)
    _next_id: int = 0

    def insert(self, text: str, session_id: str, turn_id: int) -> str:
        """
        Args:
            text: 要归档的文本。
            session_id: 会话 ID。
            turn_id: 轮次号。

        Returns:
            observation: 含 ``entry_id`` 的确认。
        """
        self._next_id += 1
        entry = MemoryEntry(text=text, session_id=session_id, turn_id=turn_id, entry_id=self._next_id)
        self.entries.append(entry)
        return (
            f"OK: archival insert id={entry.entry_id} "
            f"session={session_id} turn={turn_id}"
        )

    def search(self, query: str, top_k: int = 3) -> str:
        """
        Args:
            query: 检索查询。
            top_k: 返回条数。

        Returns:
            observation: 命中列表或未命中。
        """
        q = set(re.findall(r"\w+", query.lower()))
        scored: list[tuple[float, MemoryEntry]] = []
        for e in self.entries:
            toks = set(re.findall(r"\w+", e.text.lower()))
            score = len(q & toks) / max(len(q), 1)
            if score > 0:
                scored.append((score, e))
        scored.sort(key=lambda x: x[0], reverse=True)
        hits = scored[:top_k]
        if not hits:
            return "Observation: archival miss (page not on disk?)"
        lines = ["Observation: archival hits (page-in)"]
        for score, e in hits:
            lines.append(
                f"- id={e.entry_id} score={score:.2f} "
                f"[session={e.session_id} turn={e.turn_id}] {e.text}"
            )
        return "\n".join(lines)


@dataclass
class ConversationTurn:
    """完整对话日志中的一轮（可比主窗口更长）。"""

    turn_id: int
    role: str
    content: str


@dataclass
class ConversationLog:
    """外部对话日志；``conversation_search`` 扫全量，主窗口只保留尾部。"""

    turns: list[ConversationTurn] = field(default_factory=list)
    _next_turn: int = 0

    def add(self, role: str, content: str) -> int:
        """
        Args:
            role: ``user`` / ``assistant`` / ``tool``。
            content: 文本。

        Returns:
            turn_id: 新轮次号。
        """
        self._next_turn += 1
        self.turns.append(ConversationTurn(self._next_turn, role, content))
        return self._next_turn

    def search(self, query: str, top_k: int = 3) -> str:
        """
        Args:
            query: 查询。
            top_k: 条数。

        Returns:
            observation: 命中的历史轮次。
        """
        q = set(re.findall(r"\w+", query.lower()))
        scored: list[tuple[float, ConversationTurn]] = []
        for t in self.turns:
            toks = set(re.findall(r"\w+", t.content.lower()))
            score = len(q & toks) / max(len(q), 1)
            if score > 0:
                scored.append((score, t))
        scored.sort(key=lambda x: x[0], reverse=True)
        hits = scored[:top_k]
        if not hits:
            return "Observation: conversation miss"
        lines = ["Observation: conversation hits"]
        for score, t in hits:
            lines.append(f"- turn={t.turn_id} role={t.role} score={score:.2f}: {t.content}")
        return "\n".join(lines)

    def window(self, max_turns: int) -> list[ConversationTurn]:
        """
        Args:
            max_turns: 主上下文能装下的最近轮次数。

        Returns:
            recent: 尾部窗口（可能截断 = 溢出）。
        """
        return self.turns[-max_turns:]


@dataclass
class MemGPTState:
    """MemGPT 式双层记忆状态。"""

    session_id: str
    core: CoreMemory = field(default_factory=CoreMemory)
    archival: ArchivalStore = field(default_factory=ArchivalStore)
    conversation: ConversationLog = field(default_factory=ConversationLog)
    window_size: int = 4  # 主上下文对话槽位数（玩具偏小，易溢出）

    def main_context_chars(self) -> int:
        """
        Returns:
            n: 当前主上下文近似字符数。
        """
        win = self.conversation.window(self.window_size)
        body = "\n".join(f"{t.role}: {t.content}" for t in win)
        return len(self.core.render()) + len(body)

    def render_main_context(self) -> str:
        """
        Returns:
            prompt: 模型可见的主上下文（RAM）。
        """
        win = self.conversation.window(self.window_size)
        lines = [self.core.render(), "", "# Recency window (RAM; older turns may be paged out)"]
        if len(self.conversation.turns) > self.window_size:
            dropped = len(self.conversation.turns) - self.window_size
            lines.append(f"(overflow: {dropped} older turns not in RAM — use conversation_search)")
        for t in win:
            lines.append(f"{t.role}: {t.content}")
        return "\n".join(lines)


class MemoryTools:
    """五件记忆工具：缺页中断的入口。"""

    def __init__(self, state: MemGPTState) -> None:
        self.state = state

    def core_memory_append(self, section: str, text: str) -> str:
        """写入主上下文持久节。"""
        return self.state.core.append(section, text)

    def core_memory_replace(self, section: str, old: str, new: str) -> str:
        """编辑主上下文持久节。"""
        return self.state.core.replace(section, old, new)

    def archival_memory_insert(self, text: str) -> str:
        """写入外部档案；turn_id 取当前日志末轮。"""
        turn_id = self.state.conversation.turns[-1].turn_id if self.state.conversation.turns else 0
        return self.state.archival.insert(text, self.state.session_id, turn_id)

    def archival_memory_search(self, query: str, top_k: int = 3) -> str:
        """缺页：从磁盘调页。"""
        return self.state.archival.search(query, top_k=top_k)

    def conversation_search(self, query: str, top_k: int = 3) -> str:
        """缺页：扫完整对话日志。"""
        return self.state.conversation.search(query, top_k=top_k)

    def as_registry(self) -> dict[str, Callable[..., str]]:
        """
        Returns:
            tools: 工具名 → 可调用。
        """
        return {
            "core_memory_append": self.core_memory_append,
            "core_memory_replace": self.core_memory_replace,
            "archival_memory_insert": self.archival_memory_insert,
            "archival_memory_search": self.archival_memory_search,
            "conversation_search": self.conversation_search,
        }


@dataclass
class ToolCall:
    """一次工具调用。"""

    name: str
    args: dict[str, Any]


class LLMReply(TypedDict, total=False):
    """脚本化 LLM 一步。"""

    kind: str  # "action" | "finish"
    thought: str
    action: str
    args: dict[str, Any]
    content: str


def format_observation(tool_name: str, raw: str, max_chars: int = 800) -> str:
    """
    Args:
        tool_name: 工具名。
        raw: 原始返回。
        max_chars: 截断上限。

    Returns:
        observation: 模型可读观察。
    """
    text = raw if len(raw) <= max_chars else raw[: max_chars - 3] + "..."
    return f"[Observation from {tool_name}]\n{text}"


@dataclass
class MemGPTLoop:
    """
    记忆即中断：ReAct 循环；工具结果拼进下一轮（类比 page fault 处理完毕后 resume）。
    """

    state: MemGPTState
    tools: MemoryTools
    max_steps: int = 8

    def dispatch(self, call: ToolCall) -> str:
        """
        Args:
            call: 工具调用。

        Returns:
            observation: 结果或错误。
        """
        fn = self.tools.as_registry().get(call.name)
        if fn is None:
            return f"Error: unknown tool {call.name}"
        try:
            return fn(**call.args)
        except Exception as e:  # noqa: BLE001
            return f"Error: {type(e).__name__}: {e}"

    def run_scripted(self, user_text: str, script: list[LLMReply]) -> str:
        """
        Args:
            user_text: 用户输入。
            script: 脚本化「思考 / 行动 / 结束」。

        Returns:
            final: 最终助手回复。
        """
        self.state.conversation.add("user", user_text)
        final = ""
        for step, reply in enumerate(script):
            if step >= self.max_steps:
                break
            kind = reply.get("kind", "finish")
            if kind == "action":
                call = ToolCall(str(reply["action"]), dict(reply.get("args") or {}))
                raw = self.dispatch(call)
                obs = format_observation(call.name, raw)
                self.state.conversation.add("tool", obs)
                continue
            final = str(reply.get("content", ""))
            self.state.conversation.add("assistant", final)
            break
        return final


print("MemGPT toy runtime ready | tools =", list(MemoryTools(MemGPTState("s0")).as_registry()))


## 2. 玩具示例：溢出归档 → 缺页检索 → core 替换


In [ ]:
def demo_memgpt_paging() -> None:
    """
    1) 塞满窗口并 archival_insert 关键事实；
    2) 窗口已看不到该事实时 archival_search（缺页）；
    3) conversation_search 找回早期用户句；
    4) core_memory_replace 更新 human 偏好。
    """
    state = MemGPTState(session_id="demo-sess", window_size=3)
    tools = MemoryTools(state)
    loop = MemGPTLoop(state=state, tools=tools)

    # --- 多轮灌水，制造溢出 ---
    for i in range(4):
        state.conversation.add("user", f"filler message {i} about weather")
        state.conversation.add("assistant", f"ack weather {i}")

    # --- 重要事实：写入档案（带引用）后再被挤出窗口 ---
    important = "Project codename is Nebula; deadline is March 15."
    script_write: list[LLMReply] = [
        {
            "kind": "action",
            "thought": "fact is important → page out to archival",
            "action": "archival_memory_insert",
            "args": {"text": important},
        },
        {
            "kind": "action",
            "thought": "also pin preference in core human section",
            "action": "core_memory_append",
            "args": {"section": "human", "text": "User prefers concise answers."},
        },
        {"kind": "finish", "content": "已记下 Nebula 与截止日期。"},
    ]
    out1 = loop.run_scripted("请记住：项目代号 Nebula，截止日期 March 15。", script_write)
    assert "Nebula" in out1
    assert any("Nebula" in e.text for e in state.archival.entries)

    # 再灌水，把「记住」那几轮挤出 RAM 窗口
    for i in range(3):
        state.conversation.add("user", f"noise {i}")
        state.conversation.add("assistant", f"noise-ack {i}")

    main = state.render_main_context()
    assert "Nebula" not in main  # 主上下文已缺页
    print("=== main context after overflow (no Nebula) ===")
    print(main)
    print("chars ≈", state.main_context_chars())

    # --- 缺页：archival_search ---
    script_fault: list[LLMReply] = [
        {
            "kind": "action",
            "thought": "RAM miss → page fault → archival_memory_search",
            "action": "archival_memory_search",
            "args": {"query": "Nebula deadline", "top_k": 2},
        },
        {
            "kind": "action",
            "thought": "also scan full conversation log",
            "action": "conversation_search",
            "args": {"query": "记住 Nebula", "top_k": 2},
        },
        {
            "kind": "finish",
            "content": "代号 Nebula，截止日期 March 15（来自 archival + conversation）。",
        },
    ]
    out2 = loop.run_scripted("你还记得项目代号和截止日期吗？", script_fault)
    assert "Nebula" in out2 and "March 15" in out2

    # 最近 tool 观察应含 page-in
    tool_obs = [t.content for t in state.conversation.turns if t.role == "tool"]
    assert any("archival hits" in o for o in tool_obs)

    # --- core_replace ---
    repl = tools.core_memory_replace(
        "human", "User prefers concise answers.", "User prefers bullet answers."
    )
    assert repl.startswith("OK")
    assert "bullet" in state.core.sections["human"]

    print("\n=== after page fault answer ===")
    print(out2)
    print("\n=== core human ===")
    print(state.core.sections["human"])
    print("TOY DEMO OK")


demo_memgpt_paging()


## 3. PyTorch：可学习 archival 检索（向量外存玩具）

用词袋 → 小 MLP embedding，把 query 与档案条目投到同一空间，cosine 打分——对应笔记里「磁盘 = 向量库」的最小可训版本。


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


WORD_VOCAB: list[str] = [
    "nebula",
    "deadline",
    "march",
    "project",
    "codename",
    "weather",
    "noise",
    "prefer",
    "concise",
    "bullet",
    "alice",
    "bob",
    "meeting",
    "budget",
]


def bag(text: str) -> torch.Tensor:
    """
    Args:
        text: 英文友好的玩具文本。

    Returns:
        x: ``(V,)`` 多热词袋。
    """
    toks = set(re.findall(r"[a-z]+", text.lower()))
    x = torch.zeros(len(WORD_VOCAB), dtype=torch.float32)
    for i, w in enumerate(WORD_VOCAB):
        if w in toks:
            x[i] = 1.0
    return x


class ArchivalEmbedder(nn.Module):
    """词袋 → 低维向量（教学规模检索编码器）。"""

    def __init__(self, n_words: int = len(WORD_VOCAB), dim: int = 16) -> None:
        super().__init__()
        self.fc1 = nn.Linear(n_words, 32)
        self.fc2 = nn.Linear(32, dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: ``(V,)`` 或 ``(B, V)``。

        Returns:
            z: L2 归一化 embedding。
        """
        single = x.ndim == 1
        if single:
            x = x.unsqueeze(0)
        z = self.fc2(F.relu(self.fc1(x)))
        z = F.normalize(z, dim=-1)
        return z.squeeze(0) if single else z


def train_archival_retriever(
    pairs: list[tuple[str, str]],
    *,
    steps: int = 400,
    lr: float = 0.05,
) -> ArchivalEmbedder:
    """
    对比学习玩具：拉近 (query, 正确条目)，推远错误条目。

    Args:
        pairs: ``(query, positive_doc)``。
        steps: 优化步数。
        lr: 学习率。

    Returns:
        model: 训练后的编码器。
    """
    model = ArchivalEmbedder()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    docs = [d for _, d in pairs]
    model.train()
    for _ in range(steps):
        loss = torch.tensor(0.0)
        for q, pos in pairs:
            zq = model(bag(q))
            zp = model(bag(pos))
            neg_loss = torch.tensor(0.0)
            for d in docs:
                if d == pos:
                    continue
                zn = model(bag(d))
                neg_loss = neg_loss + F.relu(0.2 + (zq * zn).sum() - (zq * zp).sum())
            loss = loss + neg_loss
        loss = loss / max(len(pairs), 1)
        opt.zero_grad()
        loss.backward()
        opt.step()
    model.eval()
    return model


@torch.no_grad()
def retrieve(
    model: ArchivalEmbedder,
    query: str,
    corpus: list[str],
    top_k: int = 2,
) -> list[tuple[float, str]]:
    """
    Args:
        model: 编码器。
        query: 查询。
        corpus: 档案文本列表。
        top_k: 返回条数。

    Returns:
        hits: ``(score, text)`` 降序。
    """
    zq = model(bag(query))
    scored: list[tuple[float, str]] = []
    for doc in corpus:
        zd = model(bag(doc))
        scored.append((float((zq * zd).sum().item()), doc))
    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[:top_k]


def demo_pytorch_archival_retriever() -> None:
    """训练检索器：Nebula 查询应排到正确档案。"""
    torch.manual_seed(0)
    pairs = [
        ("nebula deadline?", "Project codename Nebula deadline March"),
        ("who is alice meeting?", "Alice meeting Bob about budget"),
        ("user prefer style", "User prefers bullet answers concise"),
    ]
    corpus = [d for _, d in pairs] + ["weather noise filler unrelated"]
    model = train_archival_retriever(pairs)
    hits = retrieve(model, "what is nebula project deadline", corpus, top_k=2)
    print("=== pytorch archival retrieve ===")
    for s, t in hits:
        print(f"{s:.3f}  {t}")
    assert "Nebula" in hits[0][1]
    hits2 = retrieve(model, "alice budget meeting", corpus, top_k=1)
    assert "Alice" in hits2[0][1]
    print("PYTORCH DEMO OK")


demo_pytorch_archival_retriever()


## 4. 生产级：LangChain 记忆工具 + DeepSeek

同一套 `MemGPTState` 上挂五件 ``StructuredTool``，用 ``create_agent`` 跑「记忆即中断」循环。需 ``DEEPSEEK_API_KEY``。


In [ ]:
import json
import os
import sys
from pathlib import Path
from typing import Any

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, ToolMessage
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field

sys.path.append(str(Path("../../00_Common").resolve()))
from user_tools import load_project_env  # noqa: E402

load_project_env()

MODEL = "deepseek:deepseek-v4-flash"

# 生产段共用一个可变状态（会话级单例）
PROD_STATE = MemGPTState(session_id="prod-sess", window_size=6)
PROD_TOOLS = MemoryTools(PROD_STATE)


class CoreAppendArgs(BaseModel):
    """core_memory_append 参数。"""

    section: str = Field(description="Core section name, e.g. persona or human")
    text: str = Field(description="Text to append")


class CoreReplaceArgs(BaseModel):
    """core_memory_replace 参数。"""

    section: str = Field(description="Core section name")
    old: str = Field(description="Exact substring to replace")
    new: str = Field(description="Replacement text")


class ArchivalInsertArgs(BaseModel):
    """archival_memory_insert 参数。"""

    text: str = Field(description="Fact to store in archival memory with session/turn citation")


class ArchivalSearchArgs(BaseModel):
    """archival_memory_search 参数。"""

    query: str = Field(description="Search query for archival memory")
    top_k: int = Field(default=3, description="Number of hits")


class ConversationSearchArgs(BaseModel):
    """conversation_search 参数。"""

    query: str = Field(description="Search query over full conversation log")
    top_k: int = Field(default=3, description="Number of hits")


def _make_tool(name: str, description: str, args_model: type[BaseModel], method_name: str) -> StructuredTool:
    """
    Args:
        name: 工具名。
        description: 何时使用。
        args_model: Pydantic 参数。
        method_name: ``MemoryTools`` 方法名。

    Returns:
        tool: LangChain StructuredTool。
    """

    def _run(**kwargs: Any) -> str:
        parsed = args_model(**kwargs)
        fn = getattr(PROD_TOOLS, method_name)
        return str(fn(**parsed.model_dump()))

    return StructuredTool.from_function(
        name=name,
        description=description,
        func=_run,
        args_schema=args_model,
    )


def build_memory_lc_tools() -> list[StructuredTool]:
    """
    Returns:
        tools: 五件记忆工具（闭包读当前 ``PROD_TOOLS``）。
    """
    return [
        _make_tool(
            "core_memory_append",
            "Append text to a persistent core-memory section always visible in the prompt.",
            CoreAppendArgs,
            "core_memory_append",
        ),
        _make_tool(
            "core_memory_replace",
            "Replace a substring inside a core-memory section (edit durable facts/preferences).",
            CoreReplaceArgs,
            "core_memory_replace",
        ),
        _make_tool(
            "archival_memory_insert",
            "Write an important fact to unbounded archival storage (disk). Use when RAM/window may overflow.",
            ArchivalInsertArgs,
            "archival_memory_insert",
        ),
        _make_tool(
            "archival_memory_search",
            "Page-fault style search over archival memory when the answer is not in the current window.",
            ArchivalSearchArgs,
            "archival_memory_search",
        ),
        _make_tool(
            "conversation_search",
            "Search the full conversation log (including turns paged out of the recency window).",
            ConversationSearchArgs,
            "conversation_search",
        ),
    ]


MEMORY_LC_TOOLS = build_memory_lc_tools()


def reset_prod_memory(session_id: str = "prod-sess", window_size: int = 6) -> None:
    """
    重置生产段全局记忆状态与工具列表。

    Args:
        session_id: 会话 ID。
        window_size: 主上下文对话窗口大小。
    """
    global PROD_STATE, PROD_TOOLS, MEMORY_LC_TOOLS
    PROD_STATE = MemGPTState(session_id=session_id, window_size=window_size)
    PROD_TOOLS = MemoryTools(PROD_STATE)
    MEMORY_LC_TOOLS = build_memory_lc_tools()


def get_llm(*, temperature: float = 0.0) -> Any:
    """
    Returns:
        llm: DeepSeek chat model。
    """
    if not os.getenv("DEEPSEEK_API_KEY"):
        raise RuntimeError("DEEPSEEK_API_KEY missing; copy .env.example → .env")
    return init_chat_model(
        MODEL,
        temperature=temperature,
        extra_body={"thinking": {"type": "disabled"}},
    )


def build_memgpt_agent() -> Any:
    """
    Returns:
        agent: ``create_agent`` 图（记忆工具 = 缺页处理程序）。
    """
    system = (
        "You are a MemGPT-style agent with tiered memory.\n"
        "Core memory is always visible; archival + full conversation are on disk.\n"
        "When the user shares lasting facts, call archival_memory_insert "
        "(and optionally core_memory_append for preferences).\n"
        "When you lack a fact in the current message, treat it as a page fault: "
        "call archival_memory_search and/or conversation_search, then answer.\n"
        "Reply in Chinese. Be concise."
    )
    return create_agent(
        get_llm(),
        MEMORY_LC_TOOLS,
        system_prompt=system,
    )


def format_agent_messages(messages: list[BaseMessage]) -> str:
    """
    Args:
        messages: agent 轨迹。

    Returns:
        text: 可读摘要。
    """
    lines: list[str] = []
    for m in messages:
        if isinstance(m, HumanMessage):
            lines.append(f"USER: {m.content}")
        elif isinstance(m, AIMessage):
            if m.tool_calls:
                for tc in m.tool_calls:
                    lines.append(f"ACTION: {tc['name']}({tc.get('args')})")
            if m.content:
                lines.append(f"ASSISTANT: {m.content}")
        elif isinstance(m, ToolMessage):
            lines.append(f"OBS[{m.name}]: {m.content}")
    return "\n".join(lines)


def count_tool_calls(messages: list[BaseMessage]) -> int:
    """
    Args:
        messages: 消息列表。

    Returns:
        n: 工具调用次数。
    """
    n = 0
    for m in messages:
        if isinstance(m, AIMessage) and m.tool_calls:
            n += len(m.tool_calls)
    return n


def run_memgpt_turn(user_text: str) -> dict[str, Any]:
    """
    Args:
        user_text: 用户输入。

    Returns:
        result: ``create_agent`` 返回值。
    """
    # 与玩具一致：先写入完整对话日志，便于 conversation_search
    PROD_STATE.conversation.add("user", user_text)
    agent = build_memgpt_agent()
    # 把 core + 近窗口塞进可读上下文（agent 仍靠工具读磁盘）
    hint = (
        f"{PROD_STATE.render_main_context()}\n\n"
        f"# User\n{user_text}"
    )
    result = agent.invoke({"messages": [HumanMessage(content=hint)]})
    # 记录助手最终文本
    final = ""
    for m in reversed(result["messages"]):
        if isinstance(m, AIMessage) and m.content and not m.tool_calls:
            final = m.content if isinstance(m.content, str) else str(m.content)
            break
    if final:
        PROD_STATE.conversation.add("assistant", final)
    return result


print(f"LangChain MemGPT ready | {MODEL}")


## 5. 生产示例：写入 → 窗口噪声 → 缺页找回


In [ ]:
def demo_deepseek_memgpt() -> None:
    """真实 API：归档事实后靠 search 回答；无 key 则 SKIP。"""
    if not os.getenv("DEEPSEEK_API_KEY"):
        print("SKIP production demo: DEEPSEEK_API_KEY missing")
        return

    reset_prod_memory(session_id="prod-demo", window_size=4)

    r1 = run_memgpt_turn(
        "请务必记住：项目代号是 Nebula，截止日期是 March 15。"
        "请用 archival_memory_insert 存档，并 core_memory_append 到 human 节写上偏好：简洁。"
    )
    print("=== write turn ===")
    print(format_agent_messages(r1["messages"]))
    assert count_tool_calls(r1["messages"]) >= 1
    assert len(PROD_STATE.archival.entries) >= 1 or "Nebula" in PROD_STATE.core.render()

    # 人为灌噪声，挤出窗口
    for i in range(5):
        PROD_STATE.conversation.add("user", f"noise topic {i}")
        PROD_STATE.conversation.add("assistant", f"ok noise {i}")

    r2 = run_memgpt_turn("你还记得项目代号和截止日期吗？若主上下文没有，请先 search 再回答。")
    print("\n=== page-fault turn ===")
    print(format_agent_messages(r2["messages"]))
    blob = format_agent_messages(r2["messages"]).lower()
    assert "nebula" in blob or "march" in blob
    print("\nPRODUCTION DEMO OK")


demo_deepseek_memgpt()
